# Deep Learning 016 — Backpropagation, Part 2: The "How"

Companion notebook to the lesson. We implement backpropagation **from scratch in NumPy**
for a regression network and a classification network, then verify both against Keras.

Run top to bottom on a fresh kernel. This uses the **original course dataset and the
reference implementation**, so the losses you see are the ones the lesson quotes:
`25.32 → 18.32 → 9.47 → 3.25 → 1.34`.


In [ ]:
import numpy as np
import pandas as pd


## Part A — Regression

Predict **package (LPA)** from **CGPA** and **profile score**. Both inputs sit in a
similar 0–12 range on purpose: mixed scales slow convergence.

Architecture: 2 → 2 → 1, **linear** activations, 9 trainable parameters.


In [ ]:
df = pd.DataFrame(
    [[8, 8, 4], [7, 9, 5], [6, 10, 6], [5, 12, 7]],
    columns=['cgpa', 'profile_score', 'lpa'],
)
df


### The four functions

Every backprop implementation — including the one inside Keras — is these four
responsibilities. Only `update_parameters` changes between the two tasks.


In [ ]:
def initialize_parameters(layer_dims):
    """Weights all 0.1, biases all 0 - so the Keras comparison later is exact."""
    p = {}
    for l in range(1, len(layer_dims)):
        p[f'W{l}'] = np.ones((layer_dims[l - 1], layer_dims[l])) * 0.1
        p[f'b{l}'] = np.zeros((layer_dims[l], 1))
    return p


def L_layer_forward(X, p):
    """Returns yhat AND the hidden outputs - the backward pass needs both."""
    A = X
    for l in range(1, len(p) // 2 + 1):
        A_prev = A
        A = np.dot(p[f'W{l}'].T, A_prev) + p[f'b{l}']   # linear activation
    return A, A_prev


def update_parameters(p, y, y_hat, A1, X, lr=0.001):
    """All nine gradient-descent steps, one per line.

    dL/dyhat = -2(y - yhat) already carries the minus sign, so the subtraction
    in the update rule shows up as a PLUS here. See exercise 5 about the b2 line.
    """
    e = 2 * (y - y_hat)

    p['W2'][0][0] += lr * e * A1[0][0]
    p['W2'][1][0] += lr * e * A1[1][0]
    p['b2'][0][0] = p['W2'][1][0] + lr * e        # <-- exercise 5

    p['W1'][0][0] += lr * e * p['W2'][0][0] * X[0][0]
    p['W1'][0][1] += lr * e * p['W2'][0][0] * X[1][0]
    p['b1'][0][0] += lr * e * p['W2'][0][0]

    p['W1'][1][0] += lr * e * p['W2'][1][0] * X[0][0]
    p['W1'][1][1] += lr * e * p['W2'][1][0] * X[1][0]
    p['b1'][1][0] += lr * e * p['W2'][1][0]
    return p


### Train it

Inner loop over rows, outer loop over epochs. Watch the average loss fall.


In [ ]:
params = initialize_parameters([2, 2, 1])
history = []

for epoch in range(5):
    losses = []
    for j in range(df.shape[0]):
        X = df[['cgpa', 'profile_score']].values[j].reshape(2, 1)
        y = df[['lpa']].values[j][0]

        y_hat, A1 = L_layer_forward(X, params)
        y_hat = y_hat[0][0]
        update_parameters(params, y, y_hat, A1, X)
        losses.append((y - y_hat) ** 2)

    history.append(float(np.mean(losses)))
    print(f'epoch {epoch + 1}  avg loss = {history[-1]:.4f}')


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.4))
plt.plot(range(1, len(history) + 1), history, marker='o')
plt.xlabel('epoch'); plt.ylabel('average loss')
plt.title('From-scratch regression network'); plt.grid(alpha=.3)
plt.show()


### Verify against Keras

A from-scratch implementation that merely *runs* proves nothing. Remove every source of
difference — same architecture, same starting weights via `set_weights`, optimizer
**SGD** (not the default Adam), same learning rate, same loss — and the numbers agree.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(2, activation='linear'),
    layers.Dense(1, activation='linear'),
])
model.set_weights([
    np.ones((2, 2)) * 0.1, np.zeros(2),
    np.ones((2, 1)) * 0.1, np.zeros(1),
])
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.001), loss='mean_squared_error')

Xk = df[['cgpa', 'profile_score']].values.astype(float)
yk = df['lpa'].values.astype(float)
hist = model.fit(Xk, yk, epochs=5, batch_size=1, verbose=0)

print('keras final loss   :', round(hist.history['loss'][-1], 4))
print('from-scratch final :', round(history[-1], 4))


## Part B — Classification

Same 2 → 2 → 1 architecture. **Two changes**: sigmoid on every neuron, and binary
cross-entropy as the loss. That forces a re-derivation of all nine gradients.

The key simplification:

$$\frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z}
= \frac{\hat{y}-y}{\hat{y}(1-\hat{y})} \cdot \hat{y}(1-\hat{y}) = \hat{y} - y$$

The sigmoid derivative cancels against the cross-entropy derivative, leaving
$-(y - \hat{y})$ where regression had $-2(y - \hat{y})$.


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def forward_clf(X, p):
    A1 = sigmoid(np.dot(p['W1'].T, X) + p['b1'])
    yhat = sigmoid(np.dot(p['W2'].T, A1) + p['b2'])
    return yhat, A1


def update_clf(p, y, y_hat, A1, X, lr=0.001):
    e = (y - y_hat)          # the cancellation: no factor of 2, unlike MSE

    p['W2'][0][0] += lr * e * A1[0][0]
    p['W2'][1][0] += lr * e * A1[1][0]
    p['b2'][0][0] += lr * e

    # layer 1 also picks up each hidden neuron's sigmoid derivative O(1-O)
    d1 = e * p['W2'][0][0] * A1[0][0] * (1 - A1[0][0])
    d2 = e * p['W2'][1][0] * A1[1][0] * (1 - A1[1][0])

    p['W1'][0][0] += lr * d1 * X[0][0]
    p['W1'][0][1] += lr * d1 * X[1][0]
    p['b1'][0][0] += lr * d1

    p['W1'][1][0] += lr * d2 * X[0][0]
    p['W1'][1][1] += lr * d2 * X[1][0]
    p['b1'][1][0] += lr * d2
    return p


Same four students, but now the target is **placed / not placed** rather than a
salary — this is the dataset the original notebook uses.


In [ ]:
dfc = pd.DataFrame(
    [[8, 8, 1], [7, 9, 1], [6, 10, 0], [5, 5, 0]],
    columns=['cgpa', 'profile_score', 'placed'],
)
dfc


In [ ]:
params_c = initialize_parameters([2, 2, 1])
hist_c = []

for epoch in range(50):
    losses = []
    for j in range(dfc.shape[0]):
        X = dfc[['cgpa', 'profile_score']].values[j].reshape(2, 1)
        y = dfc[['placed']].values[j][0]

        y_hat, A1 = forward_clf(X, params_c)
        y_hat = y_hat[0][0]
        update_clf(params_c, y, y_hat, A1, X)

        eps = 1e-9   # keep log() finite
        losses.append(-(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps)))

    hist_c.append(float(np.mean(losses)))
    if (epoch + 1) % 10 == 0:
        print(f'epoch {epoch + 1:3d}  avg loss = {hist_c[-1]:.4f}')


The loss barely moves — it sits just under 0.7, which is ln(2), the loss of a model
guessing 50/50. That is **not a bug**, and the lesson sees the same plateau. Two
reasons compound: four rows is almost no signal, and with every weight initialised to
the same 0.1 the two hidden neurons compute identical values, receive identical
gradients, and stay identical forever. Exercise 2 breaks that symmetry.

Build the same network in Keras and it stalls too. An implementation that matches the
framework when it *fails* is as well-verified as one that matches when it succeeds.


### Why that 0.25 matters

The hidden-layer gradient picked up a factor `O * (1 - O)`. Plot it — it never exceeds
**0.25**. One such factor per layer is the entire mechanism behind the vanishing
gradient problem in lesson 018.


In [ ]:
z = np.linspace(-6, 6, 200)

plt.figure(figsize=(6, 3.4))
plt.plot(z, sigmoid(z), label=r'$\sigma(z)$')
plt.plot(z, sigmoid(z) * (1 - sigmoid(z)), '--', label=r"$\sigma'(z)$")
plt.axhline(0.25, color='grey', lw=.8)
plt.annotate('max = 0.25', (2.2, 0.27), color='grey')
plt.legend(); plt.grid(alpha=.3); plt.xlabel('z')
plt.show()

print('ten sigmoid layers multiply the gradient by:', 0.25 ** 10)


## Exercises

1. Change the regression learning rate to `0.01`, then `0.1`. At what point does the
   loss stop converging, and why?
2. The classification network is stuck because every weight starts at 0.1, so both
   hidden neurons are identical and stay identical. Initialise with `np.random.randn`
   and raise the learning rate. Does it learn now? This is the **symmetry-breaking**
   problem, and it is why nobody initialises a real network to a constant.
3. Swap the classification loss for MSE while keeping the sigmoid. Derive the gradient
   by hand first, then confirm training gets worse. Why?
4. Add a second hidden layer. Write out the new layer-1 gradient before you code it —
   how many factors does it contain?
5. **Find the bug.** Look hard at the `b2` line in `update_parameters`:
   `p['b2'][0][0] = p['W2'][1][0] + lr * e`. It reads from `W2`, not `b2`, and it
   *overwrites* instead of accumulating. Fix it to `p['b2'][0][0] += lr * e` and re-run:
   the losses become `26.28 → 19.44 → 10.14 → 3.39 → 1.32`. This typo is in the
   widely-circulated reference notebook, and the lesson's quoted figures come from the
   buggy version — which is why it is reproduced here rather than silently corrected.
   Two things to take away: a bias error barely moves a network this small, and
   reproducing a published result is not the same as that result being correct.
